### Import

In [1]:
import os
import sys
import pickle 
import itertools
import numpy as np
import pandas as pd
import datetime as dt
import seaborn as sns
from matplotlib import pyplot as plt 
from sklearn.model_selection import train_test_split

### General parameters

In [2]:
mimiciv  = "PATH TO DATA/mimiciv(2.2)/"
path_out = "../Data/EHR/"
path_cohort = "../Data/Cohorts/"
race_path = "Extraction/MIMICIV/Data/csvExtract/"
path_timeseries = "Extraction/MIMICIV/Data/Output/"

### Split ICU-Stays to Train-Validation-Test Sets

In [3]:
column_to_read = ['subject_id', 'hadm_id', 'stay_id', 'Bins', 'age', 'race',
                  'icu_expire_flag', 'hospital_expire_flag',  'icuLos_h']

df_ehr = pd.read_csv(path_out + '0h_to_24h_data.csv', low_memory=False, index_col=False, usecols= column_to_read)
df_ehr = df_ehr.drop_duplicates()
df_ehr = df_ehr.sort_values(by=['stay_id', 'Bins'])
df_ehr = df_ehr.reset_index(drop=True)
df_ehr.head(2)

### Check Notes Availability in First 24 Hours

In [4]:
df_note_ids = pd.read_csv(path_out + 'all_note_ids.csv', low_memory=False, index_col=False)
df_note_ids = df_note_ids[['stay_id', 'Bins', 'radiology_note']]
df_note_ids.head(2)

In [5]:
df_note_ids = df_note_ids[(df_note_ids.Bins >= 0) & (df_note_ids.Bins <= 23)]
pat_has_note_24h = list(df_note_ids.stay_id.unique())

df_ehr['has_notes'] = 0
df_ehr.loc[df_ehr.stay_id.isin(pat_has_note_24h), 'has_notes'] = 1

### Check CXRs Availability in First 24 Hours

In [7]:
df_cxr_ids = pd.read_csv(path_out + 'all_cxr_ids.csv', low_memory=False, index_col=False)
df_cxr_ids = df_cxr_ids[['stay_id', 'Bins', 'cxr_image']]
df_cxr_ids.head(2)

#### Filter CXRs based on AP & PA Position

In [8]:
cxrrecord = pd.read_csv(mimiciv + "cxr/cxr-record-list.csv")
cxrmetadata = pd.read_csv(mimiciv + "cxr/mimic-cxr-2.0.0-metadata.csv")
cxr_record  = pd.merge(cxrmetadata, cxrrecord, on=['dicom_id', 'subject_id', 'study_id'], how='left')
cxr_record = cxr_record[cxr_record.ViewPosition.isin(["AP", "PA"])]

In [9]:
df_cxr_ids = df_cxr_ids[df_cxr_ids.cxr_image.isin(cxr_record.dicom_id.unique())]

In [10]:
df_cxr_ids = df_cxr_ids[(df_cxr_ids.Bins >= 0) & (df_cxr_ids.Bins <= 23)]
pat_has_cxr_24h = list(df_cxr_ids.stay_id.unique())

df_ehr['has_cxrs'] = 0
df_ehr.loc[df_ehr.stay_id.isin(pat_has_cxr_24h), 'has_cxrs'] = 1

In [12]:
df_ehr.head(3)

### Fix Age

In [14]:
df_ehr = df_ehr.drop_duplicates()
df_ehr.loc[df_ehr['age'] >= 95, 'age'] = 95
df_ehr = df_ehr[df_ehr.age > 16]

### Fix Race

In [15]:
with open(race_path + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary = pickle.load(f)

In [16]:
general_ethnicity_mapping = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNKNOWN': 'Unknown',
    'UNABLE TO OBTAIN': 'Unknown',
    'PATIENT DECLINED TO ANSWER': 'Unknown',
    
    'OTHER': 'Other',
    'MIDDLE EASTERN': 'Other',
    'MULTIPLE RACE/ETHNICITY': 'Other',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other',
    'AMERICAN INDIAN/ALASKA NATIVE': 'Other',
    
    'ASIAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - SOUTH EAST ASIAN': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'BLACK/AFRICAN': 'Black/African American',
    'BLACK/CAPE VERDEAN': 'Black/African American',
    'BLACK/AFRICAN AMERICAN': 'Black/African American',
    'BLACK/CARIBBEAN ISLAND': 'Black/African American',
    
    'SOUTH AMERICAN': 'Hispanic/Latino',
    'HISPANIC OR LATINO': 'Hispanic/Latino',
    'HISPANIC/LATINO - CUBAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - COLUMBIAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - CENTRAL AMERICAN': 'Hispanic/Latino'}

In [17]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['race'] = df['race'].map(inv_ethnicity_dict)
    
    return df

In [18]:
def categorize_ethnicity(df, new_mapping):
    
    df['race'] = df['race'].map(new_mapping)
    
    return df

In [19]:
df_ehr = replace_ethnicity_with_names(df_ehr, race_dictionary)
df_ehr = categorize_ethnicity(df_ehr, general_ethnicity_mapping)

In [20]:
def transform_race_into_id(df):
    
    dx_type = df.race.unique()
    dict_dx_key = pd.factorize(dx_type)[1]
    dict_dx_val = pd.factorize(dx_type)[0]
    dictionary  = dict(zip(dict_dx_key, dict_dx_val))
    df['race'] = df['race'].map(dictionary)
    
    return df, dictionary

In [21]:
df_ehr, race_dictionary = transform_race_into_id(df_ehr)

### Create Dovosion Label

In [22]:
df_ehr['Division_Label'] = (df_ehr['race'].astype(int)).astype(str) + "_" + (df_ehr['hospital_expire_flag'].astype(int)).astype(str)

In [23]:
df_ehr.head(3)

### Take first hours of ICU of patients with more than 24 hour LoS

In [24]:
max_rows = df_ehr.groupby('stay_id').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [25]:
def short_long_icustays(df, observation_window):
    
    df_length_o_s = df.groupby('stay_id').max()[['Bins']].reset_index()
    
    df_LoS_short = df_length_o_s[df_length_o_s.Bins <  23].reset_index(drop=True)
    df_LoS_long  = df_length_o_s[df_length_o_s.Bins == 23].reset_index(drop=True)
    
    short_stays = df[df.stay_id.isin(df_LoS_short.stay_id.unique())].copy()
    short_stays = short_stays.groupby('stay_id').head(observation_window).reset_index(drop=True)

    long_stays = df[df.stay_id.isin(df_LoS_long.stay_id.unique())].copy()
    long_stays = long_stays.groupby('stay_id').head(observation_window).reset_index(drop=True)
    
    return long_stays, short_stays

In [26]:
df_long, df_short = short_long_icustays(df_ehr, observation_window)

#### Short Stays

In [27]:
icustays_shorter_24h = list(df_short.stay_id.unique())
len(icustays_shorter_24h)

11256

#### Stays with EHR, Note, Image 

In [28]:
all_modality_df = df_long.groupby('stay_id', as_index=False).first()
all_modality_df = all_modality_df[(all_modality_df.has_cxrs == 1) & (all_modality_df.has_notes == 1)].copy()
long_icustays_with_all_modality = list(all_modality_df.stay_id.unique())
len(long_icustays_with_all_modality)

9848

In [29]:
all_modality_short_df = df_short.groupby('stay_id', as_index=False).first()
all_modality_short_df = all_modality_short_df[(all_modality_short_df.has_cxrs == 1) & (all_modality_short_df.has_notes == 1)].copy()
short_icustays_with_all_modality = list(all_modality_short_df.stay_id.unique())
len(short_icustays_with_all_modality)

989

#### Remained Stays with EHR, Note

In [30]:
allstay_remained = df_long[~df_long.stay_id.isin(long_icustays_with_all_modality)].copy()
ehr_note_df = allstay_remained[allstay_remained.has_notes == 1]
long_icustays_with_ehr_note_modality = list(ehr_note_df.stay_id.unique())
len(long_icustays_with_ehr_note_modality)

36938

In [31]:
allstay_remained_short = df_short[~df_short.stay_id.isin(short_icustays_with_all_modality)].copy()
ehr_note_short_df = allstay_remained_short[allstay_remained_short.has_notes == 1]
short_icustays_with_ehr_note_modality = list(ehr_note_short_df.stay_id.unique())
len(short_icustays_with_ehr_note_modality)

3889

#### Stays with EHR Only

In [32]:
only_ehr_df = allstay_remained[~allstay_remained.stay_id.isin(long_icustays_with_ehr_note_modality)].copy()
long_icustays_with_only_ehr_modality = list(only_ehr_df.stay_id.unique())
len(long_icustays_with_only_ehr_modality)

13301

In [33]:
only_ehr_short_df = allstay_remained_short[~allstay_remained_short.stay_id.isin(short_icustays_with_ehr_note_modality)].copy()
short_icustays_with_only_ehr_modality = list(only_ehr_short_df.stay_id.unique())
len(short_icustays_with_only_ehr_modality)

6378

### Spliting Train - Validation - Test

In [34]:
# split_label = 'Division_Label'
split_label = 'hospital_expire_flag'

In [35]:
def split_train_test(long_stays, icustay_ids, label):
    
    long_stays = long_stays[long_stays.stay_id.isin(icustay_ids)].reset_index(drop=True)
    
    indexing = long_stays[['hadm_id', label]].groupby('hadm_id').head(1)
    train_id, test_id  = train_test_split(indexing, stratify= indexing[label], test_size= 0.20, random_state= 42)
    train_id, valid_id = train_test_split(train_id, stratify= train_id[label], test_size= 0.10, random_state= 42)
    
    index_train = train_id.hadm_id.unique()
    index_valid = valid_id.hadm_id.unique()
    index_test  = test_id.hadm_id.unique()

    train_df = long_stays[long_stays['hadm_id'].isin(index_train)].copy()
    valid_df = long_stays[long_stays['hadm_id'].isin(index_valid)].copy()
    test_df  = long_stays[long_stays['hadm_id'].isin(index_test)].copy()
    
    train_df[[label]] = train_df[[label]].astype(int)
    valid_df[[label]] = valid_df[[label]].astype(int)
    test_df[[label]]  = test_df[[label]].astype(int)
    
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    
    return train_df, valid_df, test_df

In [36]:
train_ehr_txt_img, valid_ehr_txt_img, test_ehr_txt_img = split_train_test(df_long, long_icustays_with_all_modality, split_label)
train_ehr_txt , valid_ehr_txt , test_ehr_txt  = split_train_test(df_long, long_icustays_with_ehr_note_modality, split_label)
train_ehr_only, valid_ehr_only, test_ehr_only = split_train_test(df_long, long_icustays_with_only_ehr_modality, split_label)

In [37]:
concat_train_df = [train_ehr_txt_img, train_ehr_txt, train_ehr_only]
train_df = pd.concat(concat_train_df)

concat_valid_df = [valid_ehr_txt_img, valid_ehr_txt, valid_ehr_only]
valid_df = pd.concat(concat_valid_df)

concat_test_df = [test_ehr_txt_img, test_ehr_txt, test_ehr_only]
test_df  = pd.concat(concat_test_df)

train_icustays = list(train_df.stay_id.unique())
valid_icustays = list(valid_df.stay_id.unique())
test_icustays  = list(test_df.stay_id.unique())

### Select ICU-Stays

In [38]:
icu_to_hosp_map = df_long.set_index('stay_id')['hadm_id'].to_dict()

hosp_dataset_assignment = {}

def assign_dataset(icu_list, dataset_name):
    for icu_id in icu_list:
        hosp_id = icu_to_hosp_map[icu_id]
        if hosp_id not in hosp_dataset_assignment:
            hosp_dataset_assignment[hosp_id] = dataset_name

In [39]:
assign_dataset(train_icustays, 'train')
assign_dataset(valid_icustays, 'valid')
assign_dataset(test_icustays , 'test')

train_df_final = df_long[df_long['hadm_id'].map(hosp_dataset_assignment) == 'train']
valid_df_final = df_long[df_long['hadm_id'].map(hosp_dataset_assignment) == 'valid']
test_df_final  = df_long[df_long['hadm_id'].map(hosp_dataset_assignment) == 'test']

concat_train_df = [df_short, train_df_final]
train_df_final  = pd.concat(concat_train_df)

train_icustays = list(train_df_final.stay_id.unique())
valid_icustays = list(valid_df_final.stay_id.unique())
test_icustays  = list(test_df_final.stay_id.unique())

### Filter Text and Images of First 24 Hours

In [40]:
note_ids_24h  = list(df_note_ids.radiology_note.unique())
image_ids_24h = list(df_cxr_ids.cxr_image.unique())

### Combine Long and Short ICU-Stay Lists 

In [41]:
long_icustays_with_all_modality.extend(short_icustays_with_all_modality)
long_icustays_with_ehr_note_modality.extend(short_icustays_with_ehr_note_modality)
long_icustays_with_only_ehr_modality.extend(short_icustays_with_only_ehr_modality)

icustays_with_all_modality = long_icustays_with_all_modality
icustays_with_ehr_note_modality = long_icustays_with_ehr_note_modality
icustays_with_only_ehr_modality = long_icustays_with_only_ehr_modality

### Save Data

In [42]:
with open(path_cohort + "icustays_ehr_text_image", "wb") as fp:   
    pickle.dump(icustays_with_all_modality, fp)
    
with open(path_cohort + "icustays_ehr_text", "wb") as fp:   
    pickle.dump(icustays_with_ehr_note_modality, fp)
    
with open(path_cohort + "icustays_ehr_only", "wb") as fp:   
    pickle.dump(icustays_with_only_ehr_modality, fp)
    
with open(path_cohort + "short_icustays", "wb") as fp:   
    pickle.dump(icustays_shorter_24h, fp)
    
with open(path_cohort + "train_icustays", "wb") as fp:   
    pickle.dump(train_icustays, fp)
    
with open(path_cohort + "valid_icustays", "wb") as fp:   
    pickle.dump(valid_icustays, fp)
    
with open(path_cohort + "test_icustays", "wb") as fp:   
    pickle.dump(test_icustays, fp)
    
with open(path_cohort + "note_ids_24h", "wb") as fp:   
    pickle.dump(note_ids_24h, fp)
    
with open(path_cohort + "image_ids_24h", "wb") as fp:   
    pickle.dump(image_ids_24h, fp)